## Проверка гипотезы о нормальном распределении
Исходные данные: 
1) интервальный ряд распределения возраста
людей, совершивших преступления в Москве в 2021 году (л/р №1);
2) интервальный ряд распределения среднего возраста людей,
совершивших преступления в Москве в 2021 году (л/р № 2).

По критерию Пирсона при уровне значимости α = 0,05 проверить
нулевую гипотезу о нормальном распределении случайной величины:
1) возраст;
2) средний возраст.

## Гипотезы о дисперсиях
Исходные данные: две случайные выборки значений возраста
преступников (л/р № 2), результаты л/р № 1.
1. По двум выборкам, сгенерированным из совокупности значений
возраста, проверить нулевую гипотезу о равенстве дисперсий генеральных
совокупностей при уровне значимости 0,05 при конкурирующей гипотезе:
a) 𝐻𝐻1: 𝐷𝐷1 > 𝐷𝐷2
б) 𝐻𝐻1: 𝐷𝐷1 ≠ 𝐷𝐷2.
2. Взяв одну из выборок, при уровне значимости 0,05 проверить
нулевую гипотезу о равенстве генеральной дисперсии значению,
полученному в л/р № 1, при конкурирующей гипотезе:
а) 𝐻𝐻1: 𝐷𝐷(𝑥𝑥) > 𝜎𝜎2
б) 𝐻𝐻1: 𝐷𝐷(𝑥𝑥) ≠ 𝜎𝜎2
в) 𝐻𝐻1: 𝐷𝐷(𝑥𝑥) < 𝜎𝜎2


In [78]:
from StatAnalysis import *
from math import *
from numpy import *
from scipy.stats import norm
from scipy.stats import chi2
def average(row):
    average = 0
    l = copy.deepcopy(row)
    for i in range(0, len(l)):
        average += float(l[i][0]) * float(l[i][1])
    return average / N(row)

def variance(row):
    mx = 0
    mx2 = 0
    l = copy.deepcopy(row)
    
    for i in range(0, len(l)):
        mx += float(l[i][0]) * float(l[i][1])
        mx2 += (float(l[i][0])**2) * float(l[i][1])
    mx = mx / N(row)
    mx2 = mx2 / N(row)
    return mx2 - (mx**2)

def i_average(row, w, size):
    average = 0
    for i in range(len(row) - 1):
        average += ((row[i] + row[i+1]) / 2) * w[i]
    return average

def i_variance(row, w, size):
    mx = i_average(row, w, size)
    variance = 0
    for i in range(len(row) - 1):
        a = (row[i] + row[i+1]) / 2
        b = ((a - mx)**2) * w[i]
        variance += b
    return variance

def i_average_square_deviation(row, w, size):
    return sqrt(i_variance(row, w, size))

def sample_size(row, accuracy=3.0, reliability=0.95):
    z = norm.ppf((1 + reliability) / 2)
    sigma = np.sqrt(variance(row))
    n = (z * sigma / accuracy)**2
    return ceil(n)

def generate_sample(row, num_samples=36):
    n = sample_size(row)
    data = []
    for age, freq in row:
        data.extend([age] * freq)
    
    sample_means = []
    smp = []
    for _ in range(num_samples):
        sample = random.choices(data, k=n)
        smp.append(sample)
        sample_means.append(round(float(np.mean(sample)), 2))
    return [sample_means, smp]

def rowNorm(row, AVERAGE, ASD): #ASD stands for average square deviation
    out = []
    for i in range(0, len(row)):
        x = row[i]
        out.append(exp(-((x-AVERAGE)**2)/(2*ASD**2))/(ASD*sqrt(2*pi)))
    return out

def gauss_curve(row, AVERAGE, ASD):
    yo = []
    c = min(row)
    l = max(row)
    step = (l - c) / 1000
    xo = []
    while c < l:
        x = c
        xo.append(x)
        yo.append(exp(-((x-AVERAGE)**2)/(2*ASD**2))/(ASD*sqrt(2*pi)))
        c += step
    return (xo, yo)
#-----------------------------------------------------------------------------
def fz(x, y=0.0, z=1.0):
    f = []
    for i in range(len(x)):
        f.append(1 / (z * sqrt(2*pi))) * exp(- ((x[i]-y)**2.0)/(2.0*(z**2.0)) )
    return f

def gauss_dist_hypothesis(i_row, a, r): #r - число параметров

    xi = [(c[0][0] + c[0][1]) / 2 for c in i_row] #Средние xi

    xini = [] #xi перемноженные на частоты
    for i in range(len(i_row)):
        xini.append(xi[i] * i_row[i][1])

    xini2 = [] #xi в квадрате перемноженные на частоты
    for i in range(len(i_row)):
        xini2.append(xi[i]*xi[i]*i_row[i][1])

    #i_sample_mean(i_row) Выборочная средняя

    #i_standard_deviation(i_row) СКО

    zi = [] #Стандартизированные значения xi
    for i in range(len(i_row)):
        if i == 0:
            zi.append((i_row[i][0][0] - i_sample_mean(i_row)) / i_standard_deviation(i_row))
        zi.append((i_row[i][0][1] - i_sample_mean(i_row)) / i_standard_deviation(i_row))
        #zi.append((xi[i]-i_sample_mean(i_row)) / i_standard_deviation(i_row)) #Это вроде не работает, сверху всё сверяется.
    print("Стандартизированные значения xi", zi)

    zi_intervals = [] #Интервалы zi, то есть [ [zi, zi+1], [zi, zi+1]... ]
    for i in range(len(zi)-1):
        zi_intervals.append([zi[i], zi[i+1]])
    #print(zi_intervals)

    fzi = [] #Значения функции плотности нормального распределения
    zi[0] = -float("inf")
    zi[len(zi)-1] = float("inf")
    cdf_values = norm.cdf(zi)
    target_min, target_max = -0.5, 0.5 #Масштабирование в диапазон [-0.5, 0.5]
    fzi = cdf_values * (target_max - target_min) + target_min
    #for i in range(len(i_row)):
        #fzi.append(float(norm.pdf(zi[i], loc=0, scale=1)))
        #fzi.append( 1/sqrt(2*pi) * exp(-(zi[i]**2) / 2) ) #Это вроде не работает, сверху всё впринципе сверяется с погрешностями.
    print("Значения функции плотности нормального распределения", fzi)

    pi = [] #Вероятности pi попадания X в интервалы
    for i in range(len(fzi)-1):
        pi.append(float(fzi[i+1] - fzi[i]))
    print(pi)

    ni = [] #Теоретические частоты
    for i in range(len(pi)):
        ni.append(N(i_row)*pi[i])
    ##for i in range(len(i_row)):
    ##    ni.append((1*N(i_row) * fzi[i]) / i_standard_deviation(i_row))
    print("Теоретические частоты ni", ni)

    chi_obs = [] #Хи наблюдаемое
    for i in range(len(ni)):
        chi_obs.append( ((i_row[i][1] - ni[i])**2.0) / ni[i] )
    print("Значения хи наблюдаемого", chi_obs)

    k = len(i_row) - r - 2
    print(k)
    chi_crit = chi2.ppf(1 - a, k)
    #k = len(i_row) - 2 - 2 - 1 #k-критерий k = m - r - 1
    #chi_crit = chi2.ppf(0.05, k) #Хи критическое


    #######################################################

    print("Хи наблюдаемое:", sum(chi_obs), "Хи критическое:", chi_crit)
    print("Генеральная совокупность распределена нормально" if chi_crit > sum(chi_obs) else "Генеральная совокупность не распределена нормально")

with open(r"Интервальный ряд.txt", "r") as f:
    row = [
        [[int(p) for p in c.split(" ")[0].split("-")], int(c.split(" ")[1][2::])]
        for c in f.readlines()
    ]

gauss_dist_hypothesis(row, 0.05, 2)
print(row)
print("------------------------------------------------------------------------------------")
with open(r"Москва_2021.txt", "r") as f:
    d_row = [int(p.split(" ")[0]) for p in f.readlines()]
d_row = create_interval_series(d_row)
print(d_row)
i_a_row = i_generate_samples(d_row)[0][0].tolist()

print(i_a_row)

i_a_row =sorted(i_a_row)
print(i_a_row)

gauss_dist_hypothesis(i_a_row, 0.05, 2)

Стандартизированные значения xi [-2.2388607927211432, -1.73300039952423, -1.2271400063273166, -0.7212796131304032, -0.2154192199334899, 0.2904411732634235, 0.7963015664603369, 1.3021619596572502, 1.8080223528541637]
Значения функции плотности нормального распределения [-0.5        -0.4584522  -0.39011501 -0.26463125 -0.08527976  0.11426063
  0.28707161  0.40356949  0.5       ]
[0.041547798890842313, 0.06833718789457993, 0.12548376199875055, 0.17935149442078813, 0.19954038239505845, 0.17281098673725748, 0.11649787461638372, 0.09643051304633943]
Теоретические частоты ni [312.6471866535884, 514.237338906714, 944.2653090405979, 1349.6199955164307, 1501.5413775228149, 1300.4026751978624, 876.6465064882875, 725.6396106737042]
Значения хи наблюдаемого [3.839559715401059, 108.52591998157018, 209.46331867652157, 563.8971523473455, 1451.957616468578, 6.036196411931499, 2.1730737697950047, 204.65066608455993]
4
Хи наблюдаемое: 2550.5435034557026 Хи критическое: 9.487729036781154
Генеральная совок

TypeError: 'float' object is not subscriptable